# 02 — See, describe, decide: the multi-agent pipeline, end to end

```
 STEP 1 PERCEIVE   eye (1 gpt-4o call, shuffled grid, blind to sides) + ruler (OpenCV, free)
                   + code: what changes across the 12 panels / what is the same in all 12
 STEP 2 ROUND 1    6 specialists, each reads only its own facts → rule, evidence, a question for the eye
 STEP 3 AGREE?     free code check ── yes ───────────────────────────────┐
                        │ no                                             │
 STEP 4 ROUND 2    ROUND2 = ask_eye: all distinct questions → 1 more     │
                   gpt-4o call on the same grid → answers added to the   │
                   facts → the 6 specialists answer again                │
                   (or ROUND2 = debate: the agents reconsider each other) │
 STEP 5 FINAL      FINAL = judge: reviews every candidate with its agents ◄┘
                   and evidence, PICKS one (may reword it positively, never invents one)
                   (or FINAL = supervisor: the earlier synthesis + check design)
```

**Agents in two styles, everything else the same:**
- **specialist:** six agents that each read *different* evidence.
- **clone:** six identical generalists reading everything; the setting where prior work found debate does not help.

**What gets saved per problem:**
- every round
- the questions sent to the eye, and its answers
- the judge's pick, which agents proposed that rule, and the judge's reason

So every final rule traces back to named agents and panel facts.

**Switching arms:** `ARMS=main` runs only look-again + judge. `ARMS=ab` also runs debate + judge for the A/B test.
`ARMS=all` adds the clone and 20b arms. Runs resume; each problem is saved as it ends.

Why the eye is asked what it is asked: `docs/eye_perception_rules.txt`.

In [ ]:
%run ./00_common.ipynb

In [ ]:
PROBLEMS = PROBLEM_SETS[os.environ.get("PROBLEM_SET", "SMOKE")]
MAIN = ("specialist", "openai/gpt-oss-120b", "ask_eye", "judge")      # (agent style, model, round 2, final step)
ALL_RUNS = {
    "main": [MAIN],                                                          # while testing: only the main arm
    "ab":   [("specialist", "openai/gpt-oss-120b", "debate", "judge"), MAIN],  # A/B: debate vs look again
    "all":  [("specialist", "openai/gpt-oss-120b", "debate", "judge"), MAIN,
             ("clone", "openai/gpt-oss-120b", "ask_eye", "judge"), ("specialist", "openai/gpt-oss-20b", "ask_eye", "judge")],
    "v5":   [MAIN, ("specialist", "gpt-4o", "ask_eye", "judge")],   # fields-v5 reasoner column (gpt-oss-120b vs
                                                                   # gpt-4o); run once per PERCEPTION_MODE
                                                                   # (grid, perpanel) to fill the 2x2
}
RUNS = ALL_RUNS[os.environ.get("ARMS", "main")]
RERUN = False              # True = recompute problems that are already saved
MAX_ROUND2 = 1             # one second round: a debate round, or a look-again round
USE_LLM_AGREEMENT = False  # agreement after round 1: a free code check; True = the original LLM judge call

def arm_name(style, model, round2, final):
    tag = "" if PERCEPTION_MODE == "grid" else f"__{PERCEPTION_MODE}"   # keep grid names back-compatible
    return f"pipeline__{style}__{model.split('/')[-1]}__{round2}-{final}{tag}"

print(len(PROBLEMS), "problems ×", len(RUNS), "arms:", [arm_name(*r) for r in RUNS])

## Who reads what

In [ ]:
ROUTES = {   # eye object attributes · eye fields · ruler measurements   (no fact comes from both)
    "shape":    dict(obj=("type", "sides", "hole", "triangle"), keys=("notes",),
                     ruler=("convexity", "corners", "angles")),                        # fill->fill agent; symmetric->gestalt
    "contour":  dict(obj=("type", "length"), keys=("lines", "texture"),
                     ruler=("convexity", "elongation")),                               # SPLIT from shape: curve/line properties
    "size":     dict(obj=("type", "length"), keys=("same",), ruler=("size", "elongation")),
    "quantity": dict(obj=("type",), keys=("count", "lines", "same", "whole"), ruler=("count",)),   # stripped fill from obj
    "fill":     dict(obj=("fill", "type"), keys=("count", "same", "whole"), ruler=()),  # ADD; PRIMARY=fill; count carries per-kind (solid vs outline)
    "spatial":  dict(obj=("type", "pointing", "length"), keys=("relations", "whole"), ruler=("rank", "position", "elongation")),
    "topology": dict(obj=("type", "hole"), keys=("relations", "lines", "whole"), ruler=("topology",)),
    "gestalt":  dict(obj=("type", "pointing"), keys=("whole", "symmetric", "same"), ruler=("position",)),  # ADD; groups/points/collinear/symmetry
    "relation": dict(obj=("type", "fill"), keys=("relations", "same"), ruler=("rank",)),  # combiner; SAGE may prune
}
RULER_ATTRS = ("area_frac", "large", "small", "convexity", "compactness", "ncorners", "elongation", "aspect",
               "centroid_x", "centroid_y", "object_count")          # contrast only over what the ruler owns
CONTRAST_FOR = {"shape": ("convexity", "compactness", "ncorners"), "contour": ("convexity", "ncorners", "elongation"),
                "topology": (), "quantity": ("object_count",), "fill": (),
                "spatial": ("centroid_x", "centroid_y", "aspect"), "size": ("area_frac", "large", "small", "elongation", "aspect"),
                "gestalt": ("centroid_x", "centroid_y", "object_count"), "relation": ()}
GUIDE = {   # adapted from the original specialists' dimension_instructions
    "shape": "Priority: shape type and number of sides (which kinds of figure, round vs straight-edged); then holes; then corners (acute or right-angle corners, measured); then convexity (measured convexity near 1 = convex, below 0.9 = has dents).",
    "contour": "Read the line facts: open vs closed, self-crossing vs not, number of free ends and branch points, line texture, and smooth vs twisting/jagged (measured convexity of a curve near 1 = smooth). If one such property holds on all left panels and no right panel, that is the rule.",
    "fill": "Fill is your ONLY dimension. Look at each object's fill (solid vs outline) and the count per kind. Rules include: all one fill vs mixed; more solid than outline (or the reverse); or the DISTINGUISHED figure (e.g. the single solid one) has a different type by side. Name what IS filled on each side.",
    "gestalt": "Look at how the whole panel is organised, not single shapes: number of groups (clusters), whether items lie along one line or curve, whether many small dots lie densely inside vs outside an outline, what the dots form together, symmetry (a mirror axis), and nested copies. If one such global property flips between the sides, that is the rule.",
    "topology": "If a topological property (inside, touching, overlapping, joined, crossing, open or closed) holds on all left panels and on no right panel, that is a clear topology rule. The seen relations come first; the measured box check is a rough second opinion.",
    "quantity": "Read the seen counts. The measured figure count is exact but merges touching shapes. If all left panels share one count and all right panels another (e.g. one vs two), that is the rule; if the count is the same on all 12 panels, count is NOT the rule.",
    "spatial": "Measured positions are exact (x and y from 0 to 1): below 0.45 vs above 0.55 suggests left side vs right side; the same with y for top vs bottom. The long axis gives vertical vs horizontal; pointing is seen.",
    "size": "Measured sizes are exact. A clear gap between the sides means large vs small; elongation and long-axis direction give elongated vs compact and vertical vs horizontal.",
    "relation": "Compare the objects inside each panel: which is bigger, which is above, which is inside. The measured comparison names each figure by its outline (3-corner, 4-corner, round). The rule is often a relation that flips between the two sides.",
}
FOCUS = {
    "shape": "shape (form type, number of sides, corner angles, convexity, holes)",
    "contour": "contour and lines (curve vs straight, open or closed, self-crossing, free ends, branching, smooth vs jagged, line texture)",
    "size": "size (absolute and relative sizes, line length, elongation, sameness of size)",
    "quantity": "quantity (number of objects, count per kind, number of lines, number of groups, sameness of number)",
    "fill": "fill and shade (solid vs outline, how many of each kind, which figure is filled, same or mixed fill across the panel)",
    "spatial": "spatial (position, pointing direction, long-axis direction, alignment along a line, arrangement, near/far, between)",
    "topology": "topology (containment, touching, overlapping, joined, lines crossing, free ends, branching, grouping, nesting)",
    "gestalt": "grouping and gestalt (clusters, points dense inside vs outside an outline, items along a line or curve, what the dots form, symmetry, nesting)",
    "relation": "relations BETWEEN the objects in the same panel (which is bigger, above or below, inside, same or different)",
}

def routed_blocks(bp, agent, extra=None):
    """The evidence one specialist is allowed to read, for all 12 panels (plus the eye's second look, if any)."""
    r, f, cv = ROUTES[agent], vlm_fields(bp), load_cv(bp)
    recs = {}
    for pid in PIDS:
        s = eye_text(f.get(pid), obj=r["obj"], keys=r["keys"])
        if r["ruler"]:
            s += " || MEASURED: " + ruler_text(cv[pid], keys=r["ruler"])
        if extra and extra.get(pid):
            s += f" || LOOKED AGAIN: {extra[pid]}"
        recs[pid] = s
    return records_block(recs)

left, right = routed_blocks("bp0038" if "bp0038" in PROBLEMS else PROBLEMS[0], "relation")
print("what the RELATION specialist reads (first 2 left panels):\n" + "\n".join(left.splitlines()[:2]))

## Prompts and answer handling

In [ ]:
STYLE = ("Say each side as a short visual concept a person would use (3-10 words), e.g. '<property> figures' or '<object A> <relation> <object B>'. "
         "Name what IS on each side ('<X>' vs '<Y>'), not what is missing ('<X>' vs 'no <X>'), unless absence itself is the rule. "
         "Do NOT mention numbers, thresholds, coordinates, 'ruler', 'blob', 'eye', 'record' or panel labels.")
QUESTION = ("If your evidence does not settle the rule (no rule, or confidence below 0.8), add ONE question_for_eye: a question about "
            "what is visible in a single panel that would help decide, answered the same way for every panel (a number, yes/no or a "
            "few words), e.g. 'how many <objects> are there?' or 'is the <object> <relation> the <object>?'. Ask only about what "
            "a person SEES (shapes, parts, arrangement, relations), never about measurements (size values, convexity, elongation, "
            "coordinates, counts of measured figures) — those are already measured exactly. Never mention sides, left, right or "
            "panel labels. Otherwise leave it empty.")
ANSWER = (STYLE + '\n' + QUESTION + '\n' + 'Return JSON only: {{"has_rule": true, "left": "<what all 6 left panels share>", '
          '"right": "<what all 6 right panels share>", "confidence": 0.0, "evidence": "<one sentence>", "question_for_eye": ""}}')

AGENT_PROMPT = """You are {who}.
{focus}
This is a Bongard problem: the 6 LEFT panels all follow ONE hidden rule; the 6 RIGHT panels break it (usually they show the opposite).
{changes}
Evidence for each panel:
LEFT panels:
{left}
RIGHT panels:
{right}
{hint}{own}
Find the simplest rule that is true for ALL 6 left panels and false for ALL 6 right panels, using this evidence.
Check it panel by panel (L1..L6 must fit, R1..R6 must not). Prefer simple, human-readable rules.
Some panels may contain extra objects that do not matter: the rule may concern only some of the objects in each panel.
A reading can be wrong on one panel: if a rule fits 11 of the 12 panels, still propose it with lower confidence
and name the exception in your evidence. If the evidence does not separate the sides, set has_rule to false.
""" + ANSWER

REQUIRE_COUNTEREXAMPLE = True    # in debate an agent may drop its rule only by naming a panel that breaks it
DEBATE_PROMPT = """You are {who}.
{focus}
A Bongard problem: the 6 LEFT panels follow one hidden rule; the 6 RIGHT panels break it.

YOUR current answer: {own}

Rules proposed by the others, with their evidence:
{others}

Your evidence:
LEFT panels:
{left}
RIGHT panels:
{right}
{hint}
Reconsider:
  - Does your rule still hold for all 12 panels? Look for a panel in YOUR evidence that breaks it.
  - Can your rule COMBINE with another into a stronger rule that still fits all 12 panels?
Change or drop your rule ONLY if you can name a panel (L1..L6 or R1..R6) in YOUR evidence that breaks it, and put
that panel in "counterexample". Others disagreeing is not a reason to change. If you combine rules, write "combined".
If you had no rule, you may adopt one that your evidence supports.
""" + STYLE + """
Return JSON only: {{"has_rule": true, "left": "<what all 6 left panels share>", "right": "<what all 6 right panels share>",
"confidence": 0.0, "evidence": "<one sentence>", "counterexample": "<panel label, 'combined', or empty>"}}"""

JUDGE_PROMPT = """You are the judge evaluating the agents' rules for a Bongard problem.
The 6 LEFT panels follow one hidden rule; the 6 RIGHT panels break it.

Rules this round:
{table}

Do at least three agents propose the same underlying rule, even if worded differently? How many distinct rules
are there? Which agent has the strongest, best-supported rule?
Return JSON only: {{"consensus": true, "distinct_rules": 0, "dominant": "<agent name>", "reasoning": "<one sentence>"}}"""

SUP_PROMPT = """You are the supervisor writing the final answer for a Bongard problem.
The 6 LEFT panels share one hidden rule; the 6 RIGHT panels break it.

Rules proposed by the agents (most agreed-on first; each with the agent's evidence):
{table}
{judge_pick}
{instr}

Panel facts (what the panels show, in words):
LEFT panels:
{left}
RIGHT panels:
{right}

{contrast}{cv_note}

Write the single best rule. You MUST give a concrete rule — never "no rule" or "unknown". If no proposed rule fits
all 12 panels, derive one from the panel facts. Prefer the most specific rule the facts support.
""" + STYLE + """
Return JSON only: {{"left": "<what all 6 left panels share>", "right": "<what all 6 right panels share>", "confidence": 0.0}}"""
INSTR = {"voluntary": "The agents agreed; state the consensus rule precisely.",
         "forced": "The agents did not agree; weigh their rules against the panel facts."}

LOOK_AGAIN_PROMPT = """You are the supervisor of a Bongard problem: the 6 LEFT panels share one hidden rule; the 6 RIGHT panels break it.
Your rule was: LEFT = "{L}" ; RIGHT = "{R}".
A check against the panel facts found these panels do not fit:
{fails}

Panel facts:
LEFT panels:
{left}
RIGHT panels:
{right}

Look again at the panels that do not fit and write a rule that fits all 12 (a revised or a different rule).
You MUST give a concrete rule.
""" + STYLE + """
Return JSON only: {{"left": "<what all 6 left panels share>", "right": "<what all 6 right panels share>", "confidence": 0.0}}"""

NAME_PROMPT = """A Bongard rule was found: LEFT = "{L}" ; RIGHT = "{R}".
One side is described by what is MISSING. Using the panel facts, say what the panels on that side DO show instead
(e.g. "<X>" vs "<Y>", not "<X>" vs "no <X>"). If absence itself is the concept (a property that is simply present
on one side and missing on the other), keep the rule as it is.

Panel facts:
LEFT panels:
{left}
RIGHT panels:
{right}
""" + STYLE + """
Return JSON only: {{"left": "<what all 6 left panels share>", "right": "<what all 6 right panels share>", "confidence": 0.0}}"""

FINAL_PROMPT = """You are the judge for a team of specialist agents solving a Bongard problem: the 6 LEFT panels share one
hidden rule; the 6 RIGHT panels break it (usually they show the opposite concept).

The team's candidate rules (who proposed each, in which round, and their evidence):
{table}

Panel facts (what was seen and measured, in words):
LEFT panels:
{left}
RIGHT panels:
{right}

{contrast}{cv_note}

Check EACH candidate against the panel facts, panel by panel. How many agents proposed a candidate is NOT evidence that
it is right: a rule from one agent that fits all 12 panels beats a rule from three agents that fails some.
Pick the ONE candidate that fits all 12 panels best; when several fit, prefer the simplest concept a person would name.
SIMPLICITY OVER DETAIL: ignore incidental details (a hole, an extra mark, an exact count) UNLESS that detail is the only
thing that separates the two sides. Prefer the most general simple concept (e.g. "large vs small", "triangles vs circles")
over an ornate one that happens to also fit.
NAME BOTH SIDES POSITIVELY: each side must state what IS present. Never write "no X" or "non-X" — if one side is "the rest",
name the concrete property those panels actually show (e.g. "quadrangles", "circles", "small figures"). Reword the picked
rule this way without changing its meaning; do not add a new idea, and do not write a rule that no candidate states.
ALWAYS DECIDE: you must pick a candidate every time — never return no rule. If none fits perfectly, pick the one that fits
the most panels and say which panel it misses.
""" + STYLE + """
Return JSON only: {{"pick": "C<number>", "left": "<what all 6 left panels share>", "right": "<what all 6 right panels share>",
"why": "<one sentence naming the panels that decide it>"}}"""

CHECK_PROMPT = """Check a candidate Bongard rule against written facts about 12 panels.
Candidate rule: LEFT panels = "{L}" ; RIGHT panels = "{R}".

LEFT panels:
{left}
RIGHT panels:
{right}

For EACH panel decide whether its facts fit the LEFT side of the rule, the RIGHT side, or neither.
Reply json: {{"L1": "left|right|neither", "L2": "...", "L3": "...", "L4": "...", "L5": "...", "L6": "...",
"R1": "...", "R2": "...", "R3": "...", "R4": "...", "R5": "...", "R6": "..."}}"""

def agent_json(prompt, model, **kw):
    """An agent call that never crashes the pipeline: a failed agent abstains."""
    try:
        return text_json(prompt, model=model, **kw)
    except BudgetExceeded:
        raise
    except Exception as e:
        print(f"      agent call failed, abstaining: {type(e).__name__}: {str(e)[:80]}")
        return {"has_rule": False}

def fmt_answer(a, evidence=False):
    if not a["has_rule"]:
        return "no rule"
    s = f"Left = {a['left']}, Right = {a['right']} (confidence {a['confidence']:.2f})"
    return s + (f" — evidence: {a['evidence']}" if evidence and a.get("evidence") else "")

def local_consensus(hyps, high=0.95, low=0.50, group=3):
    """Deterministic judge over rule-bearing answers only."""
    ruled = {n: h for n, h in hyps.items() if h["has_rule"]}
    if not ruled:
        return False, None
    confs = {n: h["confidence"] for n, h in ruled.items()}
    dom = max(confs, key=confs.get)
    if confs[dom] >= high and all(h["confidence"] < low for n, h in hyps.items() if n != dom):
        return True, dom
    groups = []
    for n, h in ruled.items():
        key = (h["left"] + " | " + h["right"]).lower()
        for g in groups:
            ref = ruled[g[0]]
            if SequenceMatcher(None, key, (ref["left"] + " | " + ref["right"]).lower()).ratio() > 0.6:
                g.append(n); break
        else:
            groups.append([n])
    return max(len(g) for g in groups) >= group, dom

def strong_override(hyps, high=0.95, low=0.50):
    """Original route: one agent >= 0.95 confident and every other agent below 0.5 -> skip debate."""
    ruled = {n: h for n, h in hyps.items() if h["has_rule"]}
    if not ruled:
        return False
    dom = max(ruled, key=lambda n: ruled[n]["confidence"])
    return ruled[dom]["confidence"] >= high and all(h["confidence"] < low for n, h in hyps.items() if n != dom)

def check_rule(left_rule, right_rule, left, right, model):
    """Text-only check: how many of the 12 panels land on their own side of the rule, and which ones do not."""
    out = agent_json(CHECK_PROMPT.format(L=left_rule, R=right_rule or f"not {left_rule}", left=left, right=right),
                     model=model, max_tokens=1500)
    fails = []
    for side, want in (("L", "left"), ("R", "right")):
        for k in range(1, 7):
            got = str(out.get(f"{side}{k}", "")).lower()
            if not got.startswith(want):
                fails.append(f"{side}{k} fits {'the ' + got.upper() + ' side' if got in ('left', 'right') else 'neither side'}")
    return 12 - len(fails), fails

_NEGATED = re.compile(r"\b(no|not|without|lacks?|never|absent)\b|\bnon[\s\-\u2010\u2011]", re.I)
_JARGON = re.compile(r"\b(ruler|blobs?|eye|records?|measured|\d-corner|[LRP]\d{1,2})\b", re.I)

def negated(h):
    """A side described by what is missing ('no triangle', 'non-triangular')."""
    return bool(_NEGATED.search(h["left"]) or _NEGATED.search(h["right"]))

def jargon(h):
    """Rule text that talks about the pipeline's evidence instead of the picture."""
    return bool(_JARGON.search(h["left"] + " " + h["right"]))

def candidates(round1, final):
    """Round-1 and second-round rules together, near-duplicates merged. Most agreed-on first, then rules that name
    both sides positively and without jargon, then confidence."""
    pool = []
    for rnd, stage in ((1, round1), (2, final)):
        for n, h in stage.items():
            if not h["has_rule"]:
                continue
            key = (h["left"] + " | " + h["right"]).lower()
            for c in pool:
                if SequenceMatcher(None, key, c["key"]).ratio() > 0.6:
                    c["agents"].add(n); c["rounds"].add(rnd); c["confidence"] = max(c["confidence"], h["confidence"]); break
            else:
                pool.append({"key": key, "h": h, "agents": {n}, "rounds": {rnd}, "confidence": h["confidence"]})
    return sorted(pool, key=lambda c: (-len(c["agents"]), jargon(c["h"]), negated(c["h"]), -c["confidence"]))

def clearly_verified(rule, pool):
    """A free-text rule that the CV measurements verify with a clear margin (not a lucky split)."""
    pred = verify_text_rule(rule, pool)
    if not pred:
        return None
    lh = _numeric_threshold(pred, pool)[1] if pred in NUMERIC else True
    return pred if cv_rule_margin({"predicate": pred, "left_holds": lh}, pool) >= CV_MIN_MARGIN else None

## The graph (LangGraph)

In [ ]:
from langgraph.graph import StateGraph, START, END

class S(TypedDict, total=False):
    bp: str; style: str; model: str; round2: str; final: str
    blocks: dict; full: tuple; facts: tuple; changes: str
    questions: list; look: dict; look_cost: float
    pick: str; source_agents: list; source_rounds: list; why: str
    contrast: dict
    round1: dict; hyps: dict; rounds: int; changed: int; consensus: bool; strong: bool; judge_llm: object
    judge_dominant: str; switches: list
    rule: str; rule_nocheck: str; path: str; check_score: int; verified: bool; cv_rule: str; cv_margin: float

def agents_for(style):
    return list(ROUTES) if style == "specialist" else [f"analyst{i}" for i in range(1, 7)]

def who_focus(style, agent):
    if style == "specialist":
        if agent == "relation":
            return ("the relational specialist (the combiner) for visual pattern recognition",
                    f"You ONLY analyse {FOCUS[agent]}; you may combine size, position, containment and shape. {GUIDE[agent]}")
        return (f"the {agent} specialist for visual pattern recognition",
                f"You ONLY analyse {FOCUS[agent]}. Ignore all other dimensions. {GUIDE[agent]}")
    return ("a visual pattern analyst", "You may use every recorded property of the panels.")

def hint_for(s, agent):
    attrs = RULER_ATTRS if s["style"] == "clone" else CONTRAST_FOR[agent]
    return "\n" + contrast_summary(s["contrast"], attrs) + "\n" if attrs else ""

def build_blocks(s, extra=None):
    """Every agent's evidence, the full record and the judge's facts in words (with the second look, if any)."""
    full = records_block(panel_records(s["bp"], "hybrid", extra=extra))    # every fact, as the agents read it
    blocks = {a: (routed_blocks(s["bp"], a, extra) if s["style"] == "specialist" else full) for a in agents_for(s["style"])}
    return {"blocks": blocks, "full": full, "facts": records_block(panel_records(s["bp"], "hybrid", words=True, extra=extra))}

def perceive(s):
    return build_blocks(s) | {"changes": changes_text(s["bp"]), "contrast": compute_contrast(load_cv(s["bp"]), attrs=RULER_ATTRS),
                              "rounds": 0, "changed": 0, "switches": [], "questions": [], "look": {}, "look_cost": 0.0}

def ask_agents(s, own=None):
    """One round of the six agents. own = their previous answers (round 2 after a second look)."""
    hyps = {}
    for a in agents_for(s["style"]):
        who, focus = who_focus(s["style"], a)
        left, right = s["blocks"][a]
        prev = ("\nYour answer before the second look: " + fmt_answer(own[a], evidence=True)
                + "\nThe facts now include LOOKED AGAIN answers from a second look at the panels; use them.\n") if own else ""
        hyps[a] = as_answer(agent_json(AGENT_PROMPT.format(who=who, focus=focus, changes=s["changes"], left=left, right=right,
                                                           hint=hint_for(s, a), own=prev),
                                       model=s["model"], max_tokens=1500, temperature=0.3))
    return hyps

def agents(s):
    hyps = ask_agents(s)
    return {"hyps": hyps, "round1": hyps}

def judge(s):
    """Original design: an LLM judge decides consensus; the local check is the fallback and audit."""
    local, dom = local_consensus(s["hyps"])
    strong = strong_override(s["hyps"])
    llm = None
    ruled = [(n, h) for n, h in s["hyps"].items() if h["has_rule"]]
    if USE_LLM_AGREEMENT and len(ruled) >= 2 and not strong:
        table = "\n".join(f"  {n}: {fmt_answer(h, evidence=True)}" for n, h in ruled)
        out = agent_json(JUDGE_PROMPT.format(table=table), model=s["model"], max_tokens=800)
        llm = bool(out["consensus"]) if "consensus" in out else None
        dom = str(out.get("dominant") or dom or "")
    return {"consensus": strong or (llm if llm is not None else local), "strong": strong, "judge_llm": llm,
            "judge_dominant": dom if dom in s["hyps"] else ""}

def route(s):
    if s["strong"] or s["consensus"] or s["rounds"] >= MAX_ROUND2 or s.get("round2", "debate") == "none":
        return "final"
    return s.get("round2", "debate")                                  # "debate" | "ask_eye"

def _distinct(questions):
    out = []
    for q in questions:
        if q and not any(SequenceMatcher(None, q.lower(), o.lower()).ratio() > 0.75 for o in out):
            out.append(q)
    return out

def ask_eye(s):
    """LOOK AGAIN: every distinct question from round 1 goes to the eye in one call on the same shuffled grid; the
    answers are added to every agent's facts and the agents answer again."""
    questions = _distinct([h.get("question", "") for h in s["round1"].values()])
    look, kept, cost = vlm_look(s["bp"], questions)
    if not look:                                                      # nobody asked anything usable: no second round
        return {"rounds": s["rounds"] + 1, "questions": kept}
    s2 = dict(s) | build_blocks(s, extra=look)
    hyps = ask_agents(s2, own=s["hyps"])
    changed = s["changed"] + sum(hyps[a]["rule"] != s["hyps"][a]["rule"] for a in hyps)
    return {"hyps": hyps, "rounds": s["rounds"] + 1, "changed": changed, "questions": kept, "look": look,
            "look_cost": cost, "blocks": s2["blocks"], "full": s2["full"], "facts": s2["facts"]}

def debate(s):
    new, changed, log = {}, s["changed"], list(s.get("switches") or [])
    for a, own in s["hyps"].items():
        who, focus = who_focus(s["style"], a)
        left, right = s["blocks"][a]
        others = "\n".join(f"  {n}: {fmt_answer(h, evidence=True)}" for n, h in s["hyps"].items()
                           if n != a and h["has_rule"]) or "  (none)"
        raw = agent_json(DEBATE_PROMPT.format(who=who, focus=focus, own=fmt_answer(own, evidence=True), others=others,
                                              left=left, right=right, hint=hint_for(s, a)),
                         model=s["model"], max_tokens=1500, temperature=0.2)
        ans = as_answer(raw)
        cx = str(raw.get("counterexample", "") or "").strip()
        if own["has_rule"] and ans["rule"] != own["rule"]:
            kept = REQUIRE_COUNTEREXAMPLE and not cx          # a switch without a named panel is conformity: refused
            log.append({"round": s["rounds"] + 1, "agent": a, "from": own["rule"], "to": ans["rule"],
                        "counterexample": cx, "kept_own": kept})
            if kept:
                ans = own
        changed += ans["rule"] != own["rule"]
        new[a] = ans
    return {"hyps": new, "rounds": s["rounds"] + 1, "changed": changed, "switches": log}

def supervisor(s):
    """Final rule, in this order: a measured rule with a clear margin (unless confident agents' rule checks out
    instead) -> synthesis over all candidates -> a CV-verifiable candidate -> the text check -> the next candidates
    -> one look-again on the failing panels -> a narrow measured rule -> the best-scoring rule tried.
    Agent-derived rules with a negated side get one attempt to name what that side shows instead."""
    cv = load_cv(s["bp"])
    left, right = s["facts"]
    cands = candidates(s["round1"], s["hyps"])
    tried = []                                                        # (score, answer, path)

    def out(h_rule, path, score, verified, nocheck=None):
        return {"rule": h_rule, "rule_nocheck": nocheck if nocheck is not None else h_rule, "path": path,
                "check_score": score, "verified": verified,
                "cv_rule": cv_rule["nl"] if cv_rule else "", "cv_margin": round(margin, 3)}

    def check(h, path):
        score, fails = check_rule(h["left"], h["right"], left, right, s["model"])
        tried.append((score, h, path))
        return score, fails

    def finish(h, path, score, nocheck=None):
        """Ship an agent-derived rule; if a side is negated, try once to name what those panels show instead and
        keep the rewrite only if it checks at least as well."""
        if negated(h):
            named = as_answer(agent_json(NAME_PROMPT.format(L=h["left"], R=h["right"], left=left, right=right),
                                         model=s["model"], max_tokens=1000) | {"has_rule": True})
            if named["has_rule"] and named["rule"] != h["rule"] and not jargon(named):
                sc, _ = check(named, path + "+named")
                if sc >= score:
                    return out(named["rule"], path + "+named", sc, False, nocheck=nocheck if nocheck is not None else h["rule"])
        return out(h["rule"], path, score, False, nocheck=nocheck)

    cv_rule = propose_verified_rule(cv, contrast=s["contrast"])
    margin = cv_rule_margin(cv_rule, cv) if cv_rule else 0.0
    if cv_rule and margin >= CV_MIN_MARGIN:                           # 1) a clear measured rule ...
        rivals = [c for c in cands if c["confidence"] >= 0.9
                  and verify_text_rule(c["h"]["rule"], cv) != cv_rule["predicate"]][:2]
        for c in rivals:                                              #    ... unless confident agents say otherwise
            score, _ = check(c["h"], "agent-over-cv")                 #        and their rule checks out
            if score >= 11:
                return finish(c["h"], "agent-over-cv", score, nocheck=cv_rule["nl"])
        return out(cv_rule["nl"], "cv-verifier", 12, True)

    table = "\n".join(f"  [{len(c['agents'])} agree] {fmt_answer(c['h'], evidence=True)}" for c in cands) \
        or "  (no agent found a rule)"
    dom = s.get("judge_dominant") or ""
    judge_pick = f"The judge rated the {dom} agent's rule the best supported." if dom else ""
    cv_note = f"\nA measured rule splits the panels, but only narrowly: {cv_rule['nl']}" if cv_rule else ""
    sup = as_answer(agent_json(SUP_PROMPT.format(table=table, judge_pick=judge_pick,
                                                 instr=INSTR["voluntary" if s["consensus"] else "forced"],
                                                 left=left, right=right, contrast=contrast_summary(s["contrast"]),
                                                 cv_note=cv_note),
                               model=s["model"], max_tokens=1500) | {"has_rule": True})
    final = sup if sup["has_rule"] else (cands[0]["h"] if cands else sup)
    if not final["has_rule"]:
        return out(cv_rule["nl"], "cv-verifier-narrow", 12, True) if cv_rule else out("", "no-rule", 0, False)
    for h in [final] + [c["h"] for c in cands]:                       # 2) a clearly CV-verifiable candidate wins
        if clearly_verified(h["rule"], cv):
            return out(h["rule"], "search-verified", 12, True, nocheck=final["rule"])
    score, fails = check(final, "synthesis")                          # 3) check against the panel facts
    if score >= 11:
        return finish(final, "synthesis", score)
    for c in cands[:3]:                                               # 4) the most agreed-on alternatives
        if c["h"]["rule"] != final["rule"]:
            sc, _ = check(c["h"], "checked-alternative")
            if sc >= 11:
                return finish(c["h"], "checked-alternative", sc, nocheck=final["rule"])
    again = as_answer(agent_json(LOOK_AGAIN_PROMPT.format(L=final["left"], R=final["right"], fails="\n".join(fails),
                                                          left=left, right=right),
                                 model=s["model"], max_tokens=1500) | {"has_rule": True})
    if again["has_rule"]:                                             # 5) one look-again on the failing panels
        sc, _ = check(again, "look-again")
        if sc >= 11:
            return finish(again, "look-again", sc, nocheck=final["rule"])
    if cv_rule:                                                       # 6) a narrow but exact measured split
        return out(cv_rule["nl"], "cv-verifier-narrow", 12, True, nocheck=final["rule"])
    clean = [t for t in tried if not jargon(t[1])] or tried          # 7) the best-scoring rule tried (no jargon;
    best_score, best, _ = max(clean, key=lambda t: t[0])              #    ties go to the supervisor's, tried first)
    return finish(best, "unchecked-best", best_score, nocheck=final["rule"])

_PICK = re.compile(r"C\s*(\d+)", re.I)

def final_judge(s):
    """The judge reviews the whole team and PICKS one candidate rule (it may reword it positively, never invent one)."""
    cv = load_cv(s["bp"])
    cands = candidates(s["round1"], s["hyps"] if s["hyps"] != s["round1"] else {})
    cv_rule = propose_verified_rule(cv, contrast=s["contrast"])
    margin = cv_rule_margin(cv_rule, cv) if cv_rule else 0.0
    base = {"cv_rule": cv_rule["nl"] if cv_rule else "", "cv_margin": round(margin, 3), "check_score": 0}
    if not cands:
        return base | {"rule": "", "rule_nocheck": "", "path": "no-rule", "verified": False,
                       "pick": "", "source_agents": [], "source_rounds": [], "why": "no agent proposed a rule"}
    table = "\n".join(f"  C{i+1} [agents: {', '.join(sorted(c['agents']))}; round {', '.join(map(str, sorted(c['rounds'])))}; "
                      f"confidence {c['confidence']:.2f}]: {c['h']['rule']} — evidence: {c['h'].get('evidence', '')}"
                      for i, c in enumerate(cands))
    cv_note = ""
    if cv_rule:
        cv_note = (f"\nMeasurement evidence: one measured property splits all 12 panels: {cv_rule['nl']} "
                   f"({'a clear' if margin >= CV_MIN_MARGIN else 'only a narrow'} margin). Treat it as evidence, not as the answer.")
    left, right = s["facts"]
    out = agent_json(FINAL_PROMPT.format(table=table, left=left, right=right, contrast=contrast_summary(s["contrast"]),
                                         cv_note=cv_note), model=s["model"], max_tokens=1500)
    m = _PICK.search(str(out.get("pick", "")))
    idx = int(m.group(1)) - 1 if m else -1
    path = "judge"
    if not 0 <= idx < len(cands):                                     # an invalid pick: the most agreed-on candidate
        idx, path = 0, "judge-fallback"
    chosen = cands[idx]
    worded = as_answer({"has_rule": True, "left": out.get("left", ""), "right": out.get("right", "")})
    final = worded if worded["has_rule"] and not jargon(worded) else chosen["h"]
    if negated(final):                                                # "X vs non-X": name what the other side shows
        named = as_answer(agent_json(NAME_PROMPT.format(L=final["left"], R=final["right"], left=left, right=right),
                                     model=s["model"], max_tokens=1000) | {"has_rule": True})
        if named["has_rule"] and not jargon(named):
            final = named
    return base | {"rule": final["rule"], "rule_nocheck": chosen["h"]["rule"], "path": path,
                   "verified": bool(clearly_verified(final["rule"], cv)), "pick": f"C{idx+1}",
                   "source_agents": sorted(chosen["agents"]), "source_rounds": sorted(chosen["rounds"]),
                   "why": str(out.get("why", ""))[:300]}

def final(s):
    return final_judge(s) if s.get("final", "judge") == "judge" else supervisor(s)

g = StateGraph(S)
for name, fn in [("perceive", perceive), ("agents", agents), ("judge", judge), ("debate", debate),
                 ("ask_eye", ask_eye), ("final", final)]:
    g.add_node(name, fn)
g.add_edge(START, "perceive"); g.add_edge("perceive", "agents"); g.add_edge("agents", "judge")
g.add_conditional_edges("judge", route, {"debate": "debate", "ask_eye": "ask_eye", "final": "final"})
g.add_edge("debate", "judge"); g.add_edge("ask_eye", "judge"); g.add_edge("final", END)
GRAPH = g.compile()
print("graph compiled")

## Run every arm (resumable)

In [ ]:
METER.tag = "perception: 1 VLM call per problem"
for bp in PROBLEMS:
    vlm_fields(bp)                                     # cached if 01 already ran

for style, model, round2, final_step in RUNS:
    arm = arm_name(style, model, round2, final_step)
    rows = load_predictions(arm)
    METER.tag = arm
    print(f"\n=== {arm} ===")
    for bp in PROBLEMS:
        if bp in rows and not RERUN:
            continue
        before, calls0 = METER.total, METER.calls
        try:
            out = GRAPH.invoke({"bp": bp, "style": style, "model": model, "round2": round2, "final": final_step})
        except BudgetExceeded:
            raise
        except Exception as e:
            print(f"  {bp}  FAILED ({type(e).__name__}: {str(e)[:100]}) — skipped, re-run to retry")
            continue
        rows[bp] = {"bp": bp, "rule": out["rule"], "rule_nocheck": out["rule_nocheck"], "path": out["path"],
                    "round2": round2, "final": final_step, "check_score": out.get("check_score", 0),
                    "verified": out["verified"], "rounds": out["rounds"], "consensus": out["consensus"],
                    "strong_consensus": out["strong"], "judge_llm": out.get("judge_llm"),
                    "pick": out.get("pick", ""), "source_agents": out.get("source_agents", []),
                    "source_rounds": out.get("source_rounds", []), "why": out.get("why", ""),
                    "questions": out.get("questions", []), "look_again": out.get("look", {}),
                    "switches": out.get("switches", []), "cv_rule": out.get("cv_rule", ""), "cv_margin": out.get("cv_margin", 0.0),
                    "round1": out["round1"], "final_hyps": out["hyps"], "calls": METER.calls - calls0,
                    "cost_usd": METER.total - before, "perception_cost_usd": perception_cost(bp)}
        save_predictions(arm, rows)
        src = ",".join(out.get("source_agents", [])) or out["path"]
        print(f"  {bp}  [{out['path']:12s} {src:18s}] {out['rule'][:90]}")
METER.report()

## Trace one problem: what each step saw, asked and decided

In [ ]:
arm = arm_name(*RUNS[-1])
bp = "bp0038" if "bp0038" in PROBLEMS else PROBLEMS[0]
r = load_predictions(arm)[bp]
print(f"{bp}   answer: {gt(bp)}\n")
print(changes_text(bp)[:600], "...\n")
for n, h in r["round1"].items():
    print(f"  round 1  {n:9s} {fmt_answer(h)}" + (f"   ASKS: {h['question']}" if h.get("question") else ""))
print(f"\n  agreed after round 1: {r['consensus']}   second rounds: {r['rounds']} ({r['round2']})")
for q in r.get("questions", []):
    print(f"  asked the eye: {q}")
if r.get("look_again"):
    print("  e.g. L1 second look:", r["look_again"].get("left_1", "")[:200])
if r["rounds"]:
    for n, h in r["final_hyps"].items():
        print(f"  round 2  {n:9s} {fmt_answer(h)}")
for sw in r.get("switches", []):
    print(f"  debate switch {sw['agent']}: {'REFUSED (no counterexample)' if sw['kept_own'] else 'accepted, ' + sw['counterexample']}")
print(f"\n  final step: {r['final']} → {r['path']}   picked {r.get('pick')} from {r.get('source_agents')} (round {r.get('source_rounds')})")
print(f"  why: {r.get('why')}")
if r.get("cv_rule"):
    print(f"  measurement evidence: {r['cv_rule']}  (margin {r['cv_margin']:.2f})")
print(f"  final rule: {r['rule']}")